# MovieLens Collaborative Filtering

transformed parquet を読み込み、協調フィルタリング用のデータを準備する。

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
DATA_DIR = Path("./data")

movies_df_transformed = pd.read_parquet(DATA_DIR / "movies_transformed.parquet")
ratings_df_transformed = pd.read_parquet(DATA_DIR / "ratings_transformed.parquet")
users_df_transformed = pd.read_parquet(DATA_DIR / "users_transformed.parquet")
transaction_df = pd.read_parquet(DATA_DIR / "transaction.parquet")

In [3]:
movies_df_transformed.head()
ratings_df_transformed.head()
users_df_transformed.head()

,user_id,gender,age,occupation,zip_code,age_category,occupation_category,gender_F,gender_M,age_18_24,...,occupation_other_or_not_specified,occupation_programmer,occupation_retired,occupation_sales_marketing,occupation_scientist,occupation_self_employed,occupation_technician_engineer,occupation_tradesman_craftsman,occupation_unemployed,occupation_writer
0,1,F,1,10,48067,under_18,k12_student,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,M,56,16,70072,56_plus,self_employed,0,1,0,...,0,0,0,0,0,1,0,0,0,0
2,3,M,25,15,55117,25_34,scientist,0,1,0,...,0,0,0,0,1,0,0,0,0,0
3,4,M,45,7,02460,45_49,executive_managerial,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,5,M,25,20,55455,25_34,writer,0,1,0,...,0,0,0,0,0,0,0,0,0,1


In [4]:
transaction_df.head()

,user_id,movie_id,rating,rating_ge_3,rating_ge_4,rating_datetime,gender,age_category,occupation_category,gender_F,...,release_decade_1910s,release_decade_1920s,release_decade_1930s,release_decade_1940s,release_decade_1950s,release_decade_1960s,release_decade_1970s,release_decade_1980s,release_decade_1990s,release_decade_2000s
0,1,1193,5,1,1,2000-12-31 22:12:40,F,under_18,k12_student,1,...,0,0,0,0,0,0,1,0,0,0
1,1,661,3,1,0,2000-12-31 22:35:09,F,under_18,k12_student,1,...,0,0,0,0,0,0,0,0,1,0
2,1,914,3,1,0,2000-12-31 22:32:48,F,under_18,k12_student,1,...,0,0,0,0,0,1,0,0,0,0
3,1,3408,4,1,1,2000-12-31 22:04:35,F,under_18,k12_student,1,...,0,0,0,0,0,0,0,0,0,1
4,1,2355,5,1,1,2001-01-06 23:38:11,F,under_18,k12_student,1,...,0,0,0,0,0,0,0,0,1,0


## Column Definition

協調フィルタリング実装時に使う列群を整理する。

In [5]:
user_attribute_columns = [
    "gender_F",
    "gender_M",
] + [
    column for column in users_df_transformed.columns
    if (
        column.startswith("age_") or column.startswith("occupation_")
    ) and column not in ["age_category", "occupation_category"]
]

movie_attribute_columns = [
    column for column in movies_df_transformed.columns
    if column.startswith("genre_") or column.startswith("release_decade_")
]

transaction_column_config = {
    "user_readable_columns": [
        "user_id",
        "gender",
        "age_category",
        "occupation_category",
    ],
    "user_attribute_columns": user_attribute_columns,
    "movie_readable_columns": [
        "movie_id",
        "title",
        "release_decade",
        "genres",
    ],
    "movie_attribute_columns": movie_attribute_columns,
    "rating_columns": [
        "user_id",
        "movie_id",
        "rating",
        "rating_ge_3",
        "rating_ge_4",
        "rating_datetime",
    ],
}

In [6]:
transaction_column_summary = pd.DataFrame(
    [
        {"source": source, "column_name": column}
        for source, columns in transaction_column_config.items()
        for column in columns
    ]
)

transaction_column_summary

,source,column_name
0,user_readable_columns,user_id
1,user_readable_columns,gender
2,user_readable_columns,age_category
3,user_readable_columns,occupation_category
4,user_attribute_columns,gender_F
...,...,...
67,rating_columns,movie_id
68,rating_columns,rating
69,rating_columns,rating_ge_3
70,rating_columns,rating_ge_4


## 協調フィルタリングの実装

`rating_ge_3 == 1` を正例として扱い、時系列の最後の2件を validation / test に回す leave-one-out 系 split を使う。ここでは候補生成段階に絞り、`Recall@100` で `id` ベース ANN と `item` ベクトル加重平均ベース ANN を比較する。


In [7]:
%pip install gensim faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [8]:
import faiss
import numpy as np
from gensim.models import Word2Vec

In [9]:
positive_df = (
    ratings_df_transformed.loc[ratings_df_transformed["rating_ge_3"] == 1]
    .sort_values(["user_id", "rating_datetime"])
    .copy()
)

eligible_users = positive_df.groupby("user_id").filter(lambda x: len(x) >= 3)["user_id"].unique()
positive_df = positive_df[positive_df["user_id"].isin(eligible_users)].copy()

valid_df = positive_df.groupby("user_id").nth(-2).copy()
test_df = positive_df.groupby("user_id").tail(1).copy()
train_df = positive_df.drop(index=valid_df.index.union(test_df.index)).copy()

split_summary = pd.DataFrame(
    {
        "n_positive_interactions": [len(positive_df)],
        "n_train_interactions": [len(train_df)],
        "n_valid_interactions": [len(valid_df)],
        "n_test_interactions": [len(test_df)],
        "n_users": [positive_df["user_id"].nunique()],
    }
)

split_summary

,n_positive_interactions,n_train_interactions,n_valid_interactions,n_test_interactions,n_users
0,836477,824401,6038,6038,6038


In [10]:
train_sequences = (
    train_df.sort_values(["user_id", "rating_datetime"])
    .groupby("user_id")["movie_id"]
    .apply(lambda x: [str(movie_id) for movie_id in x.tolist()])
)

train_sequences = [sequence for sequence in train_sequences.tolist() if len(sequence) >= 2]
len(train_sequences)

6038

In [11]:
TOP_K = 100
ANN_RETRIEVE_K = 400
EXPERIMENT_CONFIG = {
    "embedding_dim": 32,
    "window": 10,
    "epochs": 10,
    "half_life_days": 365,
}

pd.DataFrame([EXPERIMENT_CONFIG])


,embedding_dim,window,epochs,half_life_days
0,32,10,10,365


In [12]:
def train_item2vec_model(train_sequences: list[list[str]], embedding_dim: int, window: int, epochs: int) -> Word2Vec:
    return Word2Vec(
        sentences=train_sequences,
        vector_size=embedding_dim,
        window=window,
        min_count=1,
        sg=1,
        negative=5,
        epochs=epochs,
        workers=1,
        seed=42,
    )


def build_item_artifacts(item2vec_model: Word2Vec) -> tuple[dict[int, np.ndarray], np.ndarray, faiss.IndexHNSWFlat, np.ndarray]:
    item_ids = sorted(int(item) for item in item2vec_model.wv.index_to_key)
    item_id_to_vector = {
        item_id: item2vec_model.wv[str(item_id)]
        for item_id in item_ids
    }
    item_matrix = np.vstack([item_id_to_vector[item_id] for item_id in item_ids]).astype("float32")
    faiss.normalize_L2(item_matrix)
    item_index = faiss.IndexHNSWFlat(item_matrix.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
    item_index.hnsw.efConstruction = 200
    item_index.hnsw.efSearch = 128
    item_index.add(item_matrix)
    item_id_lookup = np.array(item_ids)
    return item_id_to_vector, item_matrix, item_index, item_id_lookup


In [13]:
def build_weighted_history(user_history: pd.DataFrame, half_life_days: int) -> list[tuple[int, float]]:
    weighted_history = []
    latest_timestamp = user_history["rating_datetime"].max()

    for row in user_history.itertuples(index=False):
        days_since_latest = max((latest_timestamp - row.rating_datetime).days, 0)
        recency_weight = 0.5 ** (days_since_latest / half_life_days)
        total_weight = float(row.rating * recency_weight)
        weighted_history.append((row.movie_id, total_weight))

    return weighted_history


def build_user_vector(user_history: pd.DataFrame, item_vector_map: dict[int, np.ndarray], half_life_days: int) -> np.ndarray | None:
    weighted_vectors = []
    weights = []

    for movie_id, total_weight in build_weighted_history(user_history, half_life_days):
        vector = item_vector_map.get(movie_id)
        if vector is None:
            continue
        weighted_vectors.append(vector * total_weight)
        weights.append(total_weight)

    if not weights:
        return None

    user_vector = np.sum(weighted_vectors, axis=0) / np.sum(weights)
    user_vector = user_vector.astype("float32").reshape(1, -1)
    faiss.normalize_L2(user_vector)
    return user_vector


def recommend_with_id_ann(user_history: pd.DataFrame, seen_items: set[int], item_vector_map: dict[int, np.ndarray], item_index, item_id_lookup: np.ndarray, half_life_days: int, top_k: int = TOP_K, retrieve_k: int = ANN_RETRIEVE_K) -> list[int]:
    candidate_scores: dict[int, float] = {}

    for movie_id, history_weight in build_weighted_history(user_history, half_life_days):
        item_vector = item_vector_map.get(movie_id)
        if item_vector is None:
            continue

        query_vector = item_vector.astype("float32").reshape(1, -1)
        faiss.normalize_L2(query_vector)
        distances, indices = item_index.search(query_vector, min(item_index.ntotal, retrieve_k))

        for score, candidate_idx in zip(distances[0], indices[0]):
            candidate_item_id = int(item_id_lookup[candidate_idx])
            if candidate_item_id in seen_items:
                continue
            candidate_scores[candidate_item_id] = candidate_scores.get(candidate_item_id, 0.0) + float(score) * history_weight

    ranked_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    return [item_id for item_id, _ in ranked_candidates[:top_k]]


def recommend_with_user_vector_ann(user_history: pd.DataFrame, seen_items: set[int], item_vector_map: dict[int, np.ndarray], item_index, item_id_lookup: np.ndarray, half_life_days: int, top_k: int = TOP_K, retrieve_k: int = ANN_RETRIEVE_K) -> list[int]:
    user_vector = build_user_vector(user_history, item_vector_map, half_life_days)
    if user_vector is None:
        return []

    effective_retrieve_k = min(item_index.ntotal, max(retrieve_k, top_k + len(seen_items)))
    distances, indices = item_index.search(user_vector, effective_retrieve_k)
    ranked_item_ids = item_id_lookup[indices[0]].tolist()

    recommendations = []
    for item_id in ranked_item_ids:
        if item_id in seen_items:
            continue
        recommendations.append(int(item_id))
        if len(recommendations) >= top_k:
            break
    return recommendations


def evaluate_candidate_generator(target_items: dict[int, int], seen_items: dict[int, set[int]], history_frame: pd.DataFrame, item_vector_map: dict[int, np.ndarray], item_index, item_id_lookup: np.ndarray, half_life_days: int, generator_name: str, top_k: int = TOP_K, retrieve_k: int = ANN_RETRIEVE_K) -> pd.DataFrame:
    evaluation_rows = []

    if generator_name == "id_ann":
        generator = recommend_with_id_ann
    elif generator_name == "user_vector_ann":
        generator = recommend_with_user_vector_ann
    else:
        raise ValueError(f"Unknown generator_name: {generator_name}")

    for user_id, target_item in target_items.items():
        user_history = history_frame.loc[history_frame["user_id"] == user_id]
        if user_history.empty:
            continue
        recommendations = generator(
            user_history=user_history,
            seen_items=seen_items[user_id],
            item_vector_map=item_vector_map,
            item_index=item_index,
            item_id_lookup=item_id_lookup,
            half_life_days=half_life_days,
            top_k=top_k,
            retrieve_k=retrieve_k,
        )
        evaluation_rows.append(
            {
                "user_id": user_id,
                "target_item": target_item,
                "recall_at_100": float(target_item in recommendations),
                "n_candidates": len(recommendations),
            }
        )

    return pd.DataFrame(evaluation_rows)


In [14]:
item2vec_model = train_item2vec_model(
    train_sequences=train_sequences,
    embedding_dim=EXPERIMENT_CONFIG["embedding_dim"],
    window=EXPERIMENT_CONFIG["window"],
    epochs=EXPERIMENT_CONFIG["epochs"],
)
item_id_to_vector, item_matrix, item_index, item_id_lookup = build_item_artifacts(item2vec_model)

valid_seen_items = train_df.groupby("user_id")["movie_id"].apply(set).to_dict()
valid_items = valid_df.set_index("user_id")["movie_id"].to_dict()

valid_id_ann_df = evaluate_candidate_generator(
    target_items=valid_items,
    seen_items=valid_seen_items,
    history_frame=train_df,
    item_vector_map=item_id_to_vector,
    item_index=item_index,
    item_id_lookup=item_id_lookup,
    half_life_days=EXPERIMENT_CONFIG["half_life_days"],
    generator_name="id_ann",
)

valid_user_vector_ann_df = evaluate_candidate_generator(
    target_items=valid_items,
    seen_items=valid_seen_items,
    history_frame=train_df,
    item_vector_map=item_id_to_vector,
    item_index=item_index,
    item_id_lookup=item_id_lookup,
    half_life_days=EXPERIMENT_CONFIG["half_life_days"],
    generator_name="user_vector_ann",
)

pd.DataFrame(
    [
        {
            "split": "valid",
            "method": "id_ann",
            "recall_at_100": valid_id_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(valid_id_ann_df),
        },
        {
            "split": "valid",
            "method": "user_vector_ann",
            "recall_at_100": valid_user_vector_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(valid_user_vector_ann_df),
        },
    ]
)


,split,method,recall_at_100,n_eval_users
0,valid,id_ann,0.174561,6038
1,valid,user_vector_ann,0.160318,6038


In [15]:
train_valid_df = pd.concat([train_df, valid_df]).sort_values(["user_id", "rating_datetime"]).copy()
train_valid_sequences = (
    train_valid_df.groupby("user_id")["movie_id"]
    .apply(lambda x: [str(movie_id) for movie_id in x.tolist()])
)
train_valid_sequences = [sequence for sequence in train_valid_sequences.tolist() if len(sequence) >= 2]

train_valid_item2vec_model = train_item2vec_model(
    train_sequences=train_valid_sequences,
    embedding_dim=EXPERIMENT_CONFIG["embedding_dim"],
    window=EXPERIMENT_CONFIG["window"],
    epochs=EXPERIMENT_CONFIG["epochs"],
)
item_id_to_vector, item_matrix, item_index, item_id_lookup = build_item_artifacts(train_valid_item2vec_model)

test_seen_items = train_valid_df.groupby("user_id")["movie_id"].apply(set).to_dict()
test_items = test_df.set_index("user_id")["movie_id"].to_dict()

test_id_ann_df = evaluate_candidate_generator(
    target_items=test_items,
    seen_items=test_seen_items,
    history_frame=train_valid_df,
    item_vector_map=item_id_to_vector,
    item_index=item_index,
    item_id_lookup=item_id_lookup,
    half_life_days=EXPERIMENT_CONFIG["half_life_days"],
    generator_name="id_ann",
)

test_user_vector_ann_df = evaluate_candidate_generator(
    target_items=test_items,
    seen_items=test_seen_items,
    history_frame=train_valid_df,
    item_vector_map=item_id_to_vector,
    item_index=item_index,
    item_id_lookup=item_id_lookup,
    half_life_days=EXPERIMENT_CONFIG["half_life_days"],
    generator_name="user_vector_ann",
)

test_id_ann_df.head()


,user_id,target_item,recall_at_100,n_candidates
0,1,48,0.0,100
1,2,1917,0.0,100
2,3,2081,0.0,100
3,4,1954,1.0,100
4,5,1884,0.0,100


In [16]:
pd.DataFrame(
    [
        {
            "split": "valid",
            "method": "id_ann",
            "recall_at_100": valid_id_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(valid_id_ann_df),
        },
        {
            "split": "valid",
            "method": "user_vector_ann",
            "recall_at_100": valid_user_vector_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(valid_user_vector_ann_df),
        },
        {
            "split": "test",
            "method": "id_ann",
            "recall_at_100": test_id_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(test_id_ann_df),
        },
        {
            "split": "test",
            "method": "user_vector_ann",
            "recall_at_100": test_user_vector_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(test_user_vector_ann_df),
        },
    ]
)


,split,method,recall_at_100,n_eval_users
0,valid,id_ann,0.174561,6038
1,valid,user_vector_ann,0.160318,6038
2,test,id_ann,0.170752,6038
3,test,user_vector_ann,0.158496,6038


## 正例ヒットの重なり

2 手法が拾えた正例の重なりを split ごとに確認する。`both_hit` は両方でヒット、`id_only_hit` は `id_ann` のみ、`user_vector_only_hit` は `user_vector_ann` のみを表す。


In [17]:
def compare_hit_overlap(id_eval_df: pd.DataFrame, user_vector_eval_df: pd.DataFrame, split_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    comparison_df = (
        id_eval_df[["user_id", "target_item", "recall_at_100"]]
        .rename(columns={"recall_at_100": "id_ann_hit"})
        .merge(
            user_vector_eval_df[["user_id", "recall_at_100"]].rename(columns={"recall_at_100": "user_vector_ann_hit"}),
            on="user_id",
            how="inner",
        )
    )
    comparison_df["id_ann_hit"] = comparison_df["id_ann_hit"].astype(int)
    comparison_df["user_vector_ann_hit"] = comparison_df["user_vector_ann_hit"].astype(int)

    summary_df = pd.DataFrame(
        [
            {
                "split": split_name,
                "n_users": len(comparison_df),
                "both_hit": int(((comparison_df["id_ann_hit"] == 1) & (comparison_df["user_vector_ann_hit"] == 1)).sum()),
                "id_only_hit": int(((comparison_df["id_ann_hit"] == 1) & (comparison_df["user_vector_ann_hit"] == 0)).sum()),
                "user_vector_only_hit": int(((comparison_df["id_ann_hit"] == 0) & (comparison_df["user_vector_ann_hit"] == 1)).sum()),
                "neither_hit": int(((comparison_df["id_ann_hit"] == 0) & (comparison_df["user_vector_ann_hit"] == 0)).sum()),
            }
        ]
    )
    summary_df["union_hit_users"] = summary_df["both_hit"] + summary_df["id_only_hit"] + summary_df["user_vector_only_hit"]
    summary_df["overlap_rate_within_union"] = np.where(
        summary_df["union_hit_users"] > 0,
        summary_df["both_hit"] / summary_df["union_hit_users"],
        0.0,
    )

    difference_df = comparison_df.loc[comparison_df["id_ann_hit"] != comparison_df["user_vector_ann_hit"]].copy()
    return summary_df, difference_df


valid_overlap_summary_df, valid_overlap_diff_df = compare_hit_overlap(valid_id_ann_df, valid_user_vector_ann_df, "valid")
test_overlap_summary_df, test_overlap_diff_df = compare_hit_overlap(test_id_ann_df, test_user_vector_ann_df, "test")

pd.concat([valid_overlap_summary_df, test_overlap_summary_df], ignore_index=True)


,split,n_users,both_hit,id_only_hit,user_vector_only_hit,neither_hit,union_hit_users,overlap_rate_within_union
0,valid,6038,843,211,125,4859,1179,0.715013
1,test,6038,821,210,136,4871,1167,0.703513


In [18]:
pd.concat([
    valid_overlap_diff_df.assign(split="valid"),
    test_overlap_diff_df.assign(split="test"),
], ignore_index=True).head(20)


,user_id,target_item,id_ann_hit,user_vector_ann_hit,split
0,16,2724,0,1,valid
1,20,1240,0,1,valid
2,69,1120,0,1,valid
3,71,2712,1,0,valid
4,72,3105,1,0,valid
5,79,350,1,0,valid
6,127,1214,1,0,valid
7,141,2203,0,1,valid
8,153,3785,0,1,valid
9,174,2001,1,0,valid


## Two-Tower Retrieval

`user` と `item` を別 tower で表現する最小構成の retrieval モデルを追加する。`user tower` は `user_id embedding + user属性線形項`、`item tower` は `movie_id embedding + item属性線形項` で構成し、BPR 風の pairwise loss で学習する。評価は既存と同じ `Recall@100` を使う。


In [19]:
TWO_TOWER_CONFIG = {
    "embedding_dim": 32,
    "learning_rate": 0.03,
    "regularization": 1e-4,
    "epochs": 5,
    "max_samples_per_epoch": 200_000,
    "seed": 42,
}

pd.DataFrame([TWO_TOWER_CONFIG])


,embedding_dim,learning_rate,regularization,epochs,max_samples_per_epoch,seed
0,32,0.03,0.0001,5,200000,42


In [20]:
def prepare_two_tower_data(
    train_frame: pd.DataFrame,
    user_frame: pd.DataFrame,
    movie_frame: pd.DataFrame,
    candidate_item_ids: list[int] | None = None,
):
    train_user_items = train_frame.groupby("user_id")["movie_id"].apply(set).to_dict()
    user_ids = sorted(train_user_items)
    if candidate_item_ids is None:
        item_ids = sorted(movie_frame["movie_id"].unique().tolist())
    else:
        item_ids = sorted(candidate_item_ids)

    user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids)}
    item_to_index = {item_id: idx for idx, item_id in enumerate(item_ids)}

    user_feature_frame = (
        user_frame[["user_id", *user_attribute_columns]]
        .drop_duplicates("user_id")
        .set_index("user_id")
        .reindex(user_ids)
        .fillna(0.0)
    )
    item_feature_frame = (
        movie_frame[["movie_id", *movie_attribute_columns]]
        .drop_duplicates("movie_id")
        .set_index("movie_id")
        .reindex(item_ids)
        .fillna(0.0)
    )

    user_feature_matrix = user_feature_frame.to_numpy(dtype="float32")
    item_feature_matrix = item_feature_frame.to_numpy(dtype="float32")
    positive_pairs = [
        (user_to_index[row.user_id], item_to_index[row.movie_id])
        for row in train_frame[["user_id", "movie_id"]].itertuples(index=False)
        if row.user_id in user_to_index and row.movie_id in item_to_index
    ]
    train_seen_index = {
        user_to_index[user_id]: {item_to_index[item_id] for item_id in item_ids_set if item_id in item_to_index}
        for user_id, item_ids_set in train_user_items.items()
    }

    return {
        "user_ids": user_ids,
        "item_ids": item_ids,
        "user_to_index": user_to_index,
        "item_to_index": item_to_index,
        "user_feature_matrix": user_feature_matrix,
        "item_feature_matrix": item_feature_matrix,
        "positive_pairs": positive_pairs,
        "train_seen_index": train_seen_index,
    }


def train_two_tower_model(two_tower_data: dict, config: dict):
    rng = np.random.default_rng(config["seed"])
    embedding_dim = config["embedding_dim"]
    n_users = len(two_tower_data["user_ids"])
    n_items = len(two_tower_data["item_ids"])
    user_feature_dim = two_tower_data["user_feature_matrix"].shape[1]
    item_feature_dim = two_tower_data["item_feature_matrix"].shape[1]

    user_id_embeddings = rng.normal(0, 0.05, size=(n_users, embedding_dim)).astype("float32")
    item_id_embeddings = rng.normal(0, 0.05, size=(n_items, embedding_dim)).astype("float32")
    user_feature_weights = rng.normal(0, 0.01, size=(user_feature_dim, embedding_dim)).astype("float32")
    item_feature_weights = rng.normal(0, 0.01, size=(item_feature_dim, embedding_dim)).astype("float32")

    pair_array = np.array(two_tower_data["positive_pairs"], dtype=np.int32)
    n_samples = min(len(pair_array), config["max_samples_per_epoch"])
    learning_rate = config["learning_rate"]
    regularization = config["regularization"]

    user_feature_matrix = two_tower_data["user_feature_matrix"]
    item_feature_matrix = two_tower_data["item_feature_matrix"]
    train_seen_index = two_tower_data["train_seen_index"]

    for _ in range(config["epochs"]):
        sampled_indices = rng.integers(0, len(pair_array), size=n_samples)
        for sampled_idx in sampled_indices:
            user_idx, pos_idx = pair_array[sampled_idx]
            neg_idx = rng.integers(0, n_items)
            while neg_idx in train_seen_index[user_idx]:
                neg_idx = rng.integers(0, n_items)

            user_attr = user_feature_matrix[user_idx]
            pos_attr = item_feature_matrix[pos_idx]
            neg_attr = item_feature_matrix[neg_idx]

            user_vec = user_id_embeddings[user_idx] + user_attr @ user_feature_weights
            pos_vec = item_id_embeddings[pos_idx] + pos_attr @ item_feature_weights
            neg_vec = item_id_embeddings[neg_idx] + neg_attr @ item_feature_weights

            x_uij = float(np.dot(user_vec, pos_vec - neg_vec))
            grad = 1.0 / (1.0 + np.exp(np.clip(x_uij, -20.0, 20.0)))

            user_grad = grad * (neg_vec - pos_vec)
            pos_grad = -grad * user_vec
            neg_grad = grad * user_vec

            user_id_embeddings[user_idx] -= learning_rate * (user_grad + regularization * user_id_embeddings[user_idx])
            item_id_embeddings[pos_idx] -= learning_rate * (pos_grad + regularization * item_id_embeddings[pos_idx])
            item_id_embeddings[neg_idx] -= learning_rate * (neg_grad + regularization * item_id_embeddings[neg_idx])
            user_feature_weights -= learning_rate * (np.outer(user_attr, user_grad) + regularization * user_feature_weights)
            item_feature_weights -= learning_rate * (
                np.outer(pos_attr, pos_grad) + np.outer(neg_attr, neg_grad) + regularization * item_feature_weights
            )

    return {
        "user_id_embeddings": user_id_embeddings,
        "item_id_embeddings": item_id_embeddings,
        "user_feature_weights": user_feature_weights,
        "item_feature_weights": item_feature_weights,
    }


def build_two_tower_representations(two_tower_data: dict, two_tower_model: dict):
    user_matrix = two_tower_model["user_id_embeddings"] + two_tower_data["user_feature_matrix"] @ two_tower_model["user_feature_weights"]
    item_matrix = two_tower_model["item_id_embeddings"] + two_tower_data["item_feature_matrix"] @ two_tower_model["item_feature_weights"]
    user_matrix = user_matrix.astype("float32")
    item_matrix = item_matrix.astype("float32")
    faiss.normalize_L2(user_matrix)
    faiss.normalize_L2(item_matrix)

    item_index = faiss.IndexHNSWFlat(item_matrix.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
    item_index.hnsw.efConstruction = 200
    item_index.hnsw.efSearch = 128
    item_index.add(item_matrix)
    item_id_lookup = np.array(two_tower_data["item_ids"])
    user_vector_map = {
        user_id: user_matrix[user_idx].reshape(1, -1)
        for user_idx, user_id in enumerate(two_tower_data["user_ids"])
    }
    return user_vector_map, item_index, item_id_lookup


def evaluate_two_tower_split(
    target_items: dict[int, int],
    seen_items: dict[int, set[int]],
    user_vector_map: dict[int, np.ndarray],
    item_index,
    item_id_lookup: np.ndarray,
    top_k: int = TOP_K,
    retrieve_k: int = ANN_RETRIEVE_K,
) -> pd.DataFrame:
    rows = []
    for user_id, target_item in target_items.items():
        user_vector = user_vector_map.get(user_id)
        if user_vector is None:
            continue
        effective_retrieve_k = min(item_index.ntotal, max(retrieve_k, top_k + len(seen_items.get(user_id, set()))))
        distances, indices = item_index.search(user_vector, effective_retrieve_k)
        ranked_item_ids = item_id_lookup[indices[0]].tolist()
        recommendations = []
        for item_id in ranked_item_ids:
            if item_id in seen_items.get(user_id, set()):
                continue
            recommendations.append(int(item_id))
            if len(recommendations) >= top_k:
                break
        rows.append(
            {
                "user_id": user_id,
                "target_item": target_item,
                "recall_at_100": float(target_item in recommendations),
                "n_candidates": len(recommendations),
            }
        )
    return pd.DataFrame(rows)


In [21]:
two_tower_candidate_item_ids = sorted(
    pd.concat([train_df["movie_id"], valid_df["movie_id"], test_df["movie_id"]]).unique().tolist()
)

valid_two_tower_data = prepare_two_tower_data(
    train_frame=train_df,
    user_frame=users_df_transformed,
    movie_frame=movies_df_transformed,
    candidate_item_ids=two_tower_candidate_item_ids,
)
valid_two_tower_model = train_two_tower_model(valid_two_tower_data, TWO_TOWER_CONFIG)
valid_two_tower_user_vectors, valid_two_tower_item_index, valid_two_tower_item_lookup = build_two_tower_representations(
    valid_two_tower_data,
    valid_two_tower_model,
)
valid_two_tower_eval_df = evaluate_two_tower_split(
    target_items=valid_items,
    seen_items=valid_seen_items,
    user_vector_map=valid_two_tower_user_vectors,
    item_index=valid_two_tower_item_index,
    item_id_lookup=valid_two_tower_item_lookup,
)

test_two_tower_data = prepare_two_tower_data(
    train_frame=train_valid_df,
    user_frame=users_df_transformed,
    movie_frame=movies_df_transformed,
    candidate_item_ids=two_tower_candidate_item_ids,
)
test_two_tower_model = train_two_tower_model(test_two_tower_data, TWO_TOWER_CONFIG)
test_two_tower_user_vectors, test_two_tower_item_index, test_two_tower_item_lookup = build_two_tower_representations(
    test_two_tower_data,
    test_two_tower_model,
)
test_two_tower_eval_df = evaluate_two_tower_split(
    target_items=test_items,
    seen_items=test_seen_items,
    user_vector_map=test_two_tower_user_vectors,
    item_index=test_two_tower_item_index,
    item_id_lookup=test_two_tower_item_lookup,
)

pd.DataFrame(
    [
        {
            "split": "valid",
            "method": "two_tower",
            "recall_at_100": valid_two_tower_eval_df["recall_at_100"].mean(),
            "n_eval_users": len(valid_two_tower_eval_df),
        },
        {
            "split": "test",
            "method": "two_tower",
            "recall_at_100": test_two_tower_eval_df["recall_at_100"].mean(),
            "n_eval_users": len(test_two_tower_eval_df),
        },
    ]
)


,split,method,recall_at_100,n_eval_users
0,valid,two_tower,0.192613,6038
1,test,two_tower,0.163465,6038


In [22]:
pd.DataFrame(
    [
        {
            "split": "valid",
            "method": "id_ann",
            "recall_at_100": valid_id_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(valid_id_ann_df),
        },
        {
            "split": "valid",
            "method": "user_vector_ann",
            "recall_at_100": valid_user_vector_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(valid_user_vector_ann_df),
        },
        {
            "split": "valid",
            "method": "two_tower",
            "recall_at_100": valid_two_tower_eval_df["recall_at_100"].mean(),
            "n_eval_users": len(valid_two_tower_eval_df),
        },
        {
            "split": "test",
            "method": "id_ann",
            "recall_at_100": test_id_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(test_id_ann_df),
        },
        {
            "split": "test",
            "method": "user_vector_ann",
            "recall_at_100": test_user_vector_ann_df["recall_at_100"].mean(),
            "n_eval_users": len(test_user_vector_ann_df),
        },
        {
            "split": "test",
            "method": "two_tower",
            "recall_at_100": test_two_tower_eval_df["recall_at_100"].mean(),
            "n_eval_users": len(test_two_tower_eval_df),
        },
    ]
)


,split,method,recall_at_100,n_eval_users
0,valid,id_ann,0.174561,6038
1,valid,user_vector_ann,0.160318,6038
2,valid,two_tower,0.192613,6038
3,test,id_ann,0.170752,6038
4,test,user_vector_ann,0.158496,6038
5,test,two_tower,0.163465,6038


## 指標の見方 

leave-one-out なので各ユーザーの正解は1件だけであり、ここでは `Recall@100` は「上位100件の候補に正解 item が含まれたユーザー比率」として読める。`id_ann` は履歴 item ごとの近傍探索結果を集約する方式、`user_vector_ann` は履歴 item ベクトルの rating・recency 加重平均で user クエリを作る方式である。
